In [32]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [59]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.seconds / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [60]:
df = read_dataframe('data/yellow_tripdata_2023-01.parquet')
df_val = read_dataframe('data/yellow_tripdata_2023-02.parquet')

In [61]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3009176 entries, 0 to 3066765
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int64         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           object        
 8   DOLocationID           object        
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  airport_fee            floa

In [44]:
df.duration.describe()

count    3.066766e+06
mean     1.566900e+01
std      4.259435e+01
min     -2.920000e+01
25%      7.116667e+00
50%      1.151667e+01
75%      1.830000e+01
max      1.002918e+04
Name: duration, dtype: float64

In [46]:
((df.duration >= 1) & (df.duration <= 60)).mean()

np.float64(0.9812202822125979)

In [62]:
categorical = ['PULocationID', 'DOLocationID']
X_dicts = df[categorical].to_dict(orient='records')
X_val_dicts = df_val[categorical].to_dict(orient='records')

In [65]:
dv = DictVectorizer()
X_train = dv.fit_transform(X_dicts)
X_val = dv.transform(X_val_dicts)

In [51]:
len(dv.feature_names_)

515

In [52]:
y_train = df['duration'].values
lr = LinearRegression()
lr.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [53]:
y_pred = lr.predict(X_train)

In [54]:
x_train_error = root_mean_squared_error(y_train, y_pred)
x_train_error

7.649262444043398

In [67]:
y_val = df_val['duration'].values
y_val_pred = lr.predict(X_val)
x_val_error = root_mean_squared_error(y_val, y_val_pred)
x_val_error

7.811812105603362